# Data Filtering Strategy

## Goal
Extract conversation sequences where:
1. Someone is directly addressed
2. The addressed person responds
3. Extract conversation context around these addressing events

## Approach for AMI Corpus
1. Use addressee extraction to find addressed utterances
2. Extract actual text content from words XML files
3. Build conversation sequences: context → addressed utterance → response → follow-up
4. Filter for scenarios where addressed person actually responds

## Approach for Friends-MMC
1. Parse JSON metadata files
2. Identify turns where someone is addressed (using face annotations or speaker patterns)
3. Extract conversation windows around addressing events
4. Filter for multi-party scenarios with direct addressing

In [4]:
import xml.etree.ElementTree as ET
from pathlib import Path
import json
import re
from collections import defaultdict
from typing import List, Dict, Tuple, Optional

def load_meeting_speakers(corpus_dir):
    """Load speaker mappings from meetings.xml"""
    corpus_path = Path(corpus_dir)
    meetings_file = corpus_path / 'corpusResources' / 'meetings.xml'
    
    if not meetings_file.exists():
        return {}
    
    tree = ET.parse(meetings_file)
    root = tree.getroot()
    ns = {'nite': 'http://nite.sourceforge.net/'}
    
    meeting_speakers = {}
    
    for meeting in root.findall('.//meeting', ns):
        meeting_id = meeting.get('observation')
        
        if not meeting_id:
            continue
        
        speakers = {}
        for speaker in meeting.findall('.//speaker', ns):
            speaker_letter = speaker.get('nxt_agent')
            participant_id = speaker.get('global_name')
            role = speaker.get('role')
            
            if speaker_letter:
                speakers[speaker_letter] = {
                    'participant_id': participant_id,
                    'role': role,
                }
        
        if speakers:
            meeting_speakers[meeting_id] = speakers
    
    return meeting_speakers

def extract_text_from_words_xml(words_file: Path) -> Dict[str, Dict]:
    """Extract text content and timing from AMI words XML file"""
    if not words_file.exists():
        return {}
    
    tree = ET.parse(words_file)
    root = tree.getroot()
    ns = {'nite': 'http://nite.sourceforge.net/'}
    
    word_data = {}
    for word in root.findall('.//w', ns):
        word_id = word.get('{http://nite.sourceforge.net/}id', '')
        text = word.text if word.text else ''
        starttime = word.get('starttime', '')
        endtime = word.get('endtime', '')
        
        word_data[word_id] = {
            'text': text,
            'starttime': float(starttime) if starttime else 0.0,
            'endtime': float(endtime) if endtime else 0.0,
        }
    
    return word_data

def parse_word_reference(ref: str, word_data: Dict[str, Dict]) -> List[str]:
    """Parse word reference, handling ranges properly"""
    pattern = r'id\(([^)]+)\)'
    word_ids = re.findall(pattern, ref)
    
    if len(word_ids) == 2 and '..' in ref:
        start_id = word_ids[0]
        end_id = word_ids[1]
        
        start_match = re.search(r'(.+\.words)(\d+)$', start_id)
        end_match = re.search(r'(.+\.words)(\d+)$', end_id)
        
        if start_match and end_match:
            base = start_match.group(1)
            start_num = int(start_match.group(2))
            end_num = int(end_match.group(2))
            
            all_ids = []
            for num in range(start_num, end_num + 1):
                word_id = f"{base}{num}"
                if word_id in word_data:
                    all_ids.append(word_id)
            return all_ids
    
    return word_ids

def extract_dialogue_act_text_and_time(dact_elem, words_dir: Path, meeting_id: str, speaker: str) -> Optional[Dict]:
    """Extract text content and timing for a dialogue act"""
    ns = {'nite': 'http://nite.sourceforge.net/'}
    
    children = dact_elem.findall('.//nite:child', ns)
    if not children:
        return None
    
    words_file = words_dir / f'{meeting_id}.{speaker}.words.xml'
    word_data = extract_text_from_words_xml(words_file)
    
    all_words = []
    all_starttimes = []
    all_endtimes = []
    
    for child in children:
        href = child.get('href', '')
        if not href:
            continue
        
        word_ids = parse_word_reference(href, word_data)
        
        for word_id in word_ids:
            full_word_id = f'{meeting_id}.{speaker}.words{word_id.split("words")[-1]}' if 'words' in word_id else word_id
            
            if full_word_id in word_data:
                word_info = word_data[full_word_id]
                all_words.append(word_info['text'])
                if word_info['starttime']:
                    all_starttimes.append(word_info['starttime'])
                if word_info['endtime']:
                    all_endtimes.append(word_info['endtime'])
            elif word_id in word_data:
                word_info = word_data[word_id]
                all_words.append(word_info['text'])
                if word_info['starttime']:
                    all_starttimes.append(word_info['starttime'])
                if word_info['endtime']:
                    all_endtimes.append(word_info['endtime'])
    
    if not all_words:
        return None
    
    return {
        'text': ' '.join(all_words).strip(),
        'starttime': min(all_starttimes) if all_starttimes else 0.0,
        'endtime': max(all_endtimes) if all_endtimes else 0.0,
    }

def extract_conversation_sequences_ami(corpus_dir: str, meeting_ids: List[str] = None) -> List[Dict]:
    """Extract conversation sequences from AMI corpus with explicit addressing"""
    corpus_path = Path(corpus_dir)
    dialogue_acts_dir = corpus_path / 'dialogueActs'
    words_dir = corpus_path / 'words'
    
    meeting_speakers = load_meeting_speakers(corpus_dir)
    
    if meeting_ids is None:
        meeting_ids = []
        for da_file in dialogue_acts_dir.glob('*.dialog-act.xml'):
            meeting_id = da_file.stem.split('.')[0]
            if meeting_id not in meeting_ids:
                tree = ET.parse(da_file)
                ns = {'nite': 'http://nite.sourceforge.net/'}
                if tree.findall('.//dact[@addressee]', ns):
                    meeting_ids.append(meeting_id)
    
    sequences = []
    
    for meeting_id in meeting_ids:
        print(f"Processing {meeting_id}...")
        
        all_dacts = []
        
        for da_file in dialogue_acts_dir.glob(f'{meeting_id}.*.dialog-act.xml'):
            speaker = da_file.stem.split('.')[1]
            
            tree = ET.parse(da_file)
            ns = {'nite': 'http://nite.sourceforge.net/'}
            
            for dact in tree.findall('.//dact', ns):
                addressee = dact.get('addressee', '').strip()
                dact_data = extract_dialogue_act_text_and_time(dact, words_dir, meeting_id, speaker)
                
                if dact_data and dact_data['text']:
                    all_dacts.append({
                        'meeting_id': meeting_id,
                        'speaker': speaker,
                        'addressee': addressee.split(',') if addressee else [],
                        'text': dact_data['text'],
                        'starttime': dact_data['starttime'],
                        'endtime': dact_data['endtime'],
                        'dact_id': dact.get('{http://nite.sourceforge.net/}id', ''),
                        'has_explicit_addressee': bool(addressee),
                    })
        
        all_dacts.sort(key=lambda x: (x['starttime'], x['speaker']))
        
        i = 0
        while i < len(all_dacts):
            dact = all_dacts[i]
            
            if dact['has_explicit_addressee']:
                addressees = dact['addressee']
                addressing_speaker = dact['speaker']
                
                context = all_dacts[max(0, i-3):i]
                
                response = None
                response_idx = None
                
                for j in range(i+1, min(i+20, len(all_dacts))):
                    next_dact = all_dacts[j]
                    
                    if next_dact['speaker'] in addressees:
                        response = next_dact
                        response_idx = j
                        break
                
                if response:
                    continuation = []
                    participants = set([addressing_speaker] + addressees)
                    
                    for k in range(response_idx + 1, len(all_dacts)):
                        next_dact = all_dacts[k]
                        
                        if next_dact['has_explicit_addressee']:
                            new_addressees = set(next_dact['addressee'])
                            current_addressees = set(addressees)
                            
                            if new_addressees != current_addressees:
                                break
                        
                        if next_dact['speaker'] in participants:
                            continuation.append(next_dact)
                        else:
                            if len(continuation) > 0:
                                break
                    
                    sequences.append({
                        'meeting_id': meeting_id,
                        'context': [{'speaker': c['speaker'], 'text': c['text']} for c in context],
                        'addressing_turn': {
                            'speaker': addressing_speaker,
                            'addressees': addressees,
                            'text': dact['text'],
                            'is_explicit': True
                        },
                        'response': {
                            'speaker': response['speaker'],
                            'text': response['text']
                        },
                        'continuation': [{'speaker': c['speaker'], 'text': c['text']} for c in continuation]
                    })
                    
                    i = response_idx + len(continuation) + 1
                else:
                    i += 1
            else:
                i += 1
    
    return sequences



In [5]:
import re
from pathlib import Path
import json
from typing import List, Dict, Optional

def detect_explicit_naming(text: str, speaker_names: List[str]) -> Optional[str]:
    """Detect explicit name mention in text"""
    text_lower = text.lower()
    for name in speaker_names:
        name_lower = name.lower()
        patterns = [
            f"\\b{name_lower}\\s*[,!?]",
            f"\\b{name_lower}\\s+",
            f",\\s*{name_lower}\\s*[,!?]",
        ]
        for pattern in patterns:
            if re.search(pattern, text_lower):
                return name
    return None

def filter_friends_mmc_conversations(friends_mmc_dir: str = 'friends_mmc', 
                                     num_turns: int = 5,
                                     split: str = 'train') -> List[Dict]:
    """Filter Friends-MMC for explicit addressing scenarios"""
    metadata_file = Path(friends_mmc_dir) / f'{num_turns}_turns' / f'{split}-metadata.json'
    
    if not metadata_file.exists():
        print(f"File not found: {metadata_file}")
        return []
    
    print(f"Loading {metadata_file}...")
    with open(metadata_file, 'r') as f:
        all_conversations = json.load(f)
    
    print(f"Loaded {len(all_conversations)} conversation windows")
    
    friends_names = ['monica', 'rachel', 'phoebe', 'ross', 'chandler', 'joey', 
                     'gunther', 'carol', 'susan', 'richard', 'janice', 'mike',
                     'emily', 'julio', 'pete', 'bonnie', 'ginger', 'frank jr']
    
    filtered_sequences = []
    
    for conv_idx, conversation in enumerate(all_conversations):
        if len(conversation) < 2:
            continue
        
        all_speakers = set(turn['speaker'] for turn in conversation)
        speaker_names = list(all_speakers) + friends_names
        
        explicit_addressing_idx = None
        addressee_name = None
        addressing_speaker = None
        
        for i in range(len(conversation)):
            current_turn = conversation[i]
            current_text = current_turn.get('content', '')
            
            explicitly_named = detect_explicit_naming(current_text, speaker_names)
            
            if explicitly_named and explicitly_named != current_turn['speaker']:
                explicit_addressing_idx = i
                addressee_name = explicitly_named
                addressing_speaker = current_turn['speaker']
                break
        
        if explicit_addressing_idx is not None:
            i = explicit_addressing_idx
            current_turn = conversation[i]
            current_text = current_turn.get('content', '')
            
            context_start = max(0, i - 2)
            context = conversation[context_start:i]
            
            response = None
            response_idx = None
            
            for j in range(i + 1, len(conversation)):
                if conversation[j]['speaker'] == addressee_name:
                    response = conversation[j]
                    response_idx = j
                    break
            
            if response:
                continuation = []
                participants = {addressing_speaker, addressee_name}
                
                for k in range(response_idx + 1, len(conversation)):
                    next_turn = conversation[k]
                    next_text = next_turn.get('content', '')
                    
                    new_named = detect_explicit_naming(next_text, speaker_names)
                    if new_named and new_named != addressee_name and new_named != next_turn['speaker']:
                        break
                    
                    if next_turn['speaker'] in participants:
                        continuation.append(next_turn)
                    else:
                        if len(continuation) > 0:
                            break
                
                filtered_sequences.append({
                    'conversation_id': f"{split}_{num_turns}turns_{conv_idx}",
                    'dataset': 'friends_mmc',
                    'context': [{'speaker': c['speaker'], 'text': c['content']} for c in context],
                    'addressing_turn': {
                        'speaker': addressing_speaker,
                        'addressee': addressee_name,
                        'text': current_text,
                        'is_explicit': True,
                        'faces_present': [f[1] for f in current_turn.get('faces', [])]
                    },
                    'response': {
                        'speaker': response['speaker'],
                        'text': response['content']
                    },
                    'continuation': [{'speaker': c['speaker'], 'text': c['content']} for c in continuation]
                })
    
    print(f"Filtered to {len(filtered_sequences)} explicit addressing sequences")
    
    seen_sequences = {}
    deduplicated = []
    
    for seq in filtered_sequences:
        addr_text = seq['addressing_turn']['text']
        resp_text = seq['response']['text']
        key = (addr_text, resp_text)
        
        if key not in seen_sequences:
            seen_sequences[key] = seq
            deduplicated.append(seq)
        else:
            existing = seen_sequences[key]
            if len(seq.get('continuation', [])) > len(existing.get('continuation', [])):
                idx = deduplicated.index(existing)
                deduplicated[idx] = seq
                seen_sequences[key] = seq
    
    print(f"Deduplicated to {len(deduplicated)} unique sequences")
    return deduplicated



In [6]:
def save_filtered_conversations(sequences: List[Dict], output_file: str):
    """Save sequences to JSON file"""
    with open(output_file, 'w') as f:
        json.dump(sequences, f, indent=2)
    print(f"Saved {len(sequences)} sequences to {output_file}")

def print_sequence_example(sequence: Dict, max_context: int = 2, max_continuation: int = 5):
    """Print formatted sequence example"""
    print("=" * 70)
    print(f"Conversation ID: {sequence.get('conversation_id', sequence.get('meeting_id', 'N/A'))}")
    print(f"Dataset: {sequence.get('dataset', 'ami')}")
    print()
    
    context = sequence.get('context', [])[-max_context:]
    if context:
        print("Context (before addressing):")
        for turn in context:
            print(f"  [{turn['speaker']}]: {turn['text'][:100]}...")
        print()
    
    addr = sequence.get('addressing_turn', {})
    print("Explicit Addressing Turn:")
    print(f"  Speaker: {addr.get('speaker', 'N/A')}")
    if 'addressees' in addr:
        print(f"  Explicitly addressed: {', '.join(addr['addressees'])}")
    elif 'addressee' in addr:
        print(f"  Explicitly addressed: {addr['addressee']}")
    if 'faces_present' in addr:
        print(f"  People present: {', '.join(addr['faces_present'])}")
    print(f"  Text: {addr.get('text', 'N/A')[:150]}...")
    print()
    
    resp = sequence.get('response', {})
    print("Response (from addressee):")
    print(f"  Speaker: {resp.get('speaker', 'N/A')}")
    print(f"  Text: {resp.get('text', 'N/A')[:150]}...")
    print()
    
    continuation = sequence.get('continuation', [])
    if continuation:
        print(f"Continuation ({len(continuation)} turns - conversation continues with same addressee):")
        for turn in continuation[:max_continuation]:
            print(f"  [{turn['speaker']}]: {turn['text'][:100]}...")
        if len(continuation) > max_continuation:
            print(f"  ... and {len(continuation) - max_continuation} more turns")
    else:
        print("Continuation: None (conversation ended after response)")
    print("=" * 70)
    print()

print("=" * 70)
print("FILTERING AMI CORPUS")
print("=" * 70)
ami_sequences = extract_conversation_sequences_ami(
    'ami_public_manual_1.6.2',
    meeting_ids=['ES2008a', 'ES2008b', 'ES2009c', 'ES2009d']
)
print(f"\nExtracted {len(ami_sequences)} sequences from AMI corpus\n")

print("=" * 70)
print("FILTERING FRIENDS-MMC")
print("=" * 70)
friends_sequences_5 = filter_friends_mmc_conversations(num_turns=5, split='train')
friends_sequences_8 = filter_friends_mmc_conversations(num_turns=8, split='train')
all_friends_sequences = friends_sequences_5 + friends_sequences_8
print(f"\nExtracted {len(all_friends_sequences)} sequences from Friends-MMC\n")

all_sequences = ami_sequences + all_friends_sequences
save_filtered_conversations(all_sequences, 'filtered_conversations.json')

if all_sequences:
    print("\n" + "=" * 70)
    print("EXAMPLE SEQUENCES")
    print("=" * 70 + "\n")
    for seq in all_sequences[:3]:
        print_sequence_example(seq)



FILTERING AMI CORPUS
Processing ES2008a...
Processing ES2008b...
Processing ES2009c...
Processing ES2009d...

Extracted 611 sequences from AMI corpus

FILTERING FRIENDS-MMC
Loading friends_mmc/5_turns/train-metadata.json...
Loaded 13584 conversation windows
Filtered to 3596 explicit addressing sequences
Deduplicated to 1611 unique sequences
Loading friends_mmc/8_turns/train-metadata.json...
Loaded 8730 conversation windows
Filtered to 3664 explicit addressing sequences
Deduplicated to 1312 unique sequences

Extracted 2923 sequences from Friends-MMC

Saved 3534 sequences to filtered_conversations.json

EXAMPLE SEQUENCES

Conversation ID: ES2008a
Dataset: ami

Context (before addressing):
  [A]: do a little discussion...
  [A]: and close ,...

Explicit Addressing Turn:
  Speaker: A
  Explicitly addressed: D, C, B
  Text: since we only have twenty five minutes ....

Response (from addressee):
  Speaker: B
  Text: Alima ....

Continuation: None (conversation ended after response)

Conversa